# Fine-Tune Granite on SDG Data

This notebook explains the **supervised fine-tuning (SFT)** step and runs the production script `fine_tune_model.py`.

**Keep the script for long GPU jobs.** Use this notebook to configure, understand each step, and launch the run from OpenShift AI Workbench.

## Where this fits

```text
sdg/sdg_flow.py
    ↓ synthetic_training_data.jsonl
fine-tuning/fine_tune_model.py   ← this step
    ↓ s3://models/finetuned/custom/v1/
pipelines/inner_loop/
pipelines/outer_loop/
```

| Item | Value |
|------|-------|
| Base model | `ibm-granite/granite-3.0-8b-instruct` (8B, ~16 GB bf16) |
| Input data | SDG JSONL from `sdg_hub/synthetic_training_data.jsonl` |
| Output (local) | `./granite-fine-tuned-checkpoints` |
| Output (S3) | `s3://models/finetuned/custom/v1/` |

## Prerequisites

1. Run **SDG** first (`sdg/sdg_flow.py`) to produce synthetic training data.
2. Granite base model cached locally (offline mode enabled in the script).
3. GPU with enough VRAM (script targets ~24 GB, e.g. L4).
4. MinIO/S3 credentials and endpoint for upload (optional: set `UPLOAD_TO_S3=false`).

## Step 1 — Environment setup

Forces Hugging Face **offline** mode and limits CPU threads to avoid tokenizer/multiprocessing issues on the workbench.

Configured in the script:
- `HF_DATASETS_OFFLINE`, `TRANSFORMERS_OFFLINE`, `HF_HUB_OFFLINE`
- `TOKENIZERS_PARALLELISM=false`

## Step 2 — Load and format SDG dataset

Reads JSONL rows and maps flexible field names (`question`/`instruction`, `answer`/`response`) into Granite chat format:

```text
<|system|> ...
<|user|> Context: ... Question: ...
<|assistant|> answer
```

Fails fast if SDG output is missing.

## Step 3 — Tokenization

Tokenizes conversations (max length 512) and sets `labels = input_ids` for causal language modeling loss.

## Step 4 — Load model (VRAM-safe)

Loads Granite 8B in **bfloat16** on GPU and **freezes most layers**. Only `lm_head` and `embed_tokens` are trainable — lightweight SFT to fit ~24 GB GPU.

## Step 5 — Training configuration

| Setting | Value |
|---------|-------|
| Epochs | 3 |
| Batch size | 2 (gradient accumulation 2) |
| Learning rate | 2e-5 |
| Precision | bf16 + gradient checkpointing |
| Checkpoint during train | `save_strategy="no"` (saved explicitly after train) |

## Step 6 — Train

Runs Hugging Face `Trainer.train()`. This is the long step (~16 GB model on GPU).

## Step 7 — Save locally

After training, persists:
- model weights
- tokenizer
- `training_metadata.json` (base model, dataset path, S3 destination)

Local path: `./granite-fine-tuned-checkpoints`

## Step 8 — Push to S3 / MinIO

Uploads the full artifact directory to:

```text
s3://models/finetuned/custom/v1/
```

This path is the **input reference** for the Kubeflow inner-loop pipeline. Expect ~15–16 GB upload for the 8B model.

## Configure S3 on OpenShift AI (before running)

Create an **S3 connection** in OpenShift AI Workbench (Settings / Connections). It should inject:

| Variable | Example |
|----------|---------|
| `AWS_S3_ENDPOINT` | MinIO route URL |
| `AWS_ACCESS_KEY_ID` | MinIO access key |
| `AWS_SECRET_ACCESS_KEY` | MinIO secret key |
| `AWS_S3_BUCKET` | `models` |

Optional project env vars: `S3_PREFIX=finetuned/custom/v1/`, `UPLOAD_TO_S3=true`

Do **not** use shell `export` — configure in OpenShift AI.

In [ ]:
import os

required = ["AWS_S3_ENDPOINT", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_BUCKET"]
missing = [name for name in required if not os.getenv(name)]

if missing:
    print("Missing (configure S3 connection in OpenShift AI):", ", ".join(missing))
else:
    bucket = os.getenv("AWS_S3_BUCKET")
    prefix = os.getenv("S3_PREFIX", "finetuned/custom/v1/")
    print("S3 target:", f"s3://{bucket}/{prefix}")

## Pre-flight check (optional)

Verify SDG training data exists before starting the long GPU run.

In [ ]:
from pathlib import Path

sdg_dataset = Path("/opt/app-root/src/knowledge-base-ai-assistant/sdg_hub/synthetic_training_data.jsonl")

if sdg_dataset.exists():
    line_count = sum(1 for _ in sdg_dataset.open())
    print(f"OK: found SDG dataset with {line_count} line(s)")
else:
    print(f"Missing: {sdg_dataset} — run sdg/sdg_flow.py first")

## Run fine-tuning

Executes the full pipeline in `fine_tune_model.py`. Training + S3 upload run in one shot.

In [ ]:
%run fine_tune_model.py

## Next step

After a successful run, continue with the **inner loop** pipeline:

`pipelines/inner_loop/run_inner_loop_pipeline.ipynb`

It reads from `s3://models/finetuned/custom/v1/` and stages candidates for the outer loop.